In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [4]:
user_id = "user_001"
application_context = "personal_assistant"

# namespace 정의
namespace = (user_id, application_context)

In [5]:
store.put(
    namespace,
    "memory_001",
    {
        "facts": [
            "사용자는 아아를 선호함",
            "사용자는 매일 아침 7시에 일어남",
        ],
        "language": "Korean",
    },
)

In [6]:
item = store.get(namespace, "memory_001")
print("저장된 메모리:", item.value)

저장된 메모리: {'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}


In [7]:
items = store.search(namespace)

In [9]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-05-30T21:45:37.301341+00:00', updated_at='2026-05-30T21:45:37.301341+00:00', score=None)]

In [10]:
store.put(
    namespace,
    "memory_002",
    {
        "facts": [
            "사용자는 랭체인 공부를 좋아함",
            "학습용 챗봇 프로젝트를 진행 중",
        ]
    },
)

In [11]:
item = store.get(namespace, "memory_002")
print("저장된 메모리:", item.value)

저장된 메모리: {'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행 중']}


In [12]:
items = store.search(namespace)

In [13]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-05-30T21:45:37.301341+00:00', updated_at='2026-05-30T21:45:37.301341+00:00', score=None),
 Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행 중']}, created_at='2026-05-30T21:46:07.875691+00:00', updated_at='2026-05-30T21:46:07.875691+00:00', score=None)]

In [14]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

In [15]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_memory(request, handler):
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = "기록된 정보 없음"
    if memories:
        # 검색된 메모리들을 텍스트 변환
        extracted_facts = []
        for item in memories:
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
        memory_content = "\n- ".join(extracted_facts)

    system_message=f"사용자 관련 장기 메모리 : {memory_content}"
    request = request.override(system_prompt=system_message)
    return handler(request)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    store=store, # store 연결
    context_schema=Context,
    middleware=[inject_memory] # 미들웨어 장착
)

In [17]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

In [18]:
response

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='ba03df37-e284-4a70-8604-422fb2b3f344'),
  AIMessage(content=[{'type': 'text', 'text': '사용자님에 대해 제가 기억하고 있는 정보는 다음과 같습니다.\n\n*   **음료 취향:** 아아(아이스 아메리카노)를 선호하십니다.\n*   **일상 패턴:** 매일 아침 7시에 기상하십니다.\n*   **관심사:** 랭체인(LangChain) 공부에 관심이 많으십니다.\n*   **진행 중인 프로젝트:** 현재 학습용 챗봇 프로젝트를 진행하고 계십니다.\n\n혹시 이 내용 외에 더 기억해 두었으면 하는 정보가 있으신가요? 프로젝트 진행 상황이나 오늘 아침 루틴은 어떠셨는지 궁금하네요!', 'extras': {'signature': 'EjQKMgEMOdbHKT6EO4bLxRs9+yrHY0xGP9xFDGi6dAaIaA99WNeLnLtlX0mIswhujCodx6MW'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ada-3eed-7ed3-b8db-02a49e9e55b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 148, 'total_tokens': 212, 'input_token_details': {'cache_read': 0}})]}